In [7]:
# ============================================================
# MetaDiv_sintax — USER SETTINGS
# Modify only this cell
# ============================================================

# Genetic marker:
#   "ITS"
#   "16S"
#   "COI"
# "CO1" is also accepted as an alias for "COI".
MODE = "ITS"

# FASTA files to classify.
# Write the filenames exactly as they appear inside:
#   MetaDiv_Builder/input/<MODE>/
#
# Examples:
# FASTA_FILES = ["ATLASMXB.fasta"]
# FASTA_FILES = ["ATLASMXB.fasta", "DIAMOND.fasta", "GF2S.fasta"]
#
# Subfolders are also accepted, for example:
# FASTA_FILES = ["dataset_A/sequences.fasta"]
FASTA_FILES = [
    "ATLASMXBC_red_sequences.fasta",
]

# Exact reference database filename stored inside:
#   MetaDiv_Builder/databases/taxonomic_reference_db/
#
# Examples:
# ITS  -> "SINTAX_EUKARYOME_ITS_v2.0.udb"
# 16S  -> "SINTAX_16S_ITGDB_seq.udb"
# COI  -> "SINTAX_MIDORI2_LONGEST_NUC_GB271_CO1.udb"
REFERENCE_DB_NAME = "SINTAX_EUK_ITS_v2.0.fasta"

# Basic SINTAX parameters
SINTAX_CUTOFF = 0.8
STRAND = "both"
THREADS = 16

# If False, FASTA files with an existing <name>_sintax.txt
# are skipped. Set True to overwrite them.
OVERWRITE_EXISTING = True

# Docker image containing VSEARCH/SINTAX
DOCKER_IMAGE = "pipecraft/vsearch:2.30.4-pc1.2.0"

# Automatically pull the image if it is not available locally.
AUTO_PULL_IMAGE = True


In [ ]:
# ============================================================
# MetaDiv_sintax
# VSEARCH/SINTAX taxonomic classification utility
# for MetaDiv Builder
#
# Expected repository structure:
#
# MetaDiv_Builder/
# ├── input/
# │   ├── ITS/
# │   ├── 16S/
# │   └── COI/
# ├── databases/
# │   └── taxonomic_reference_db/
# └── Utilities/
#     └── MetaDiv_sintax/
#         └── MetaDiv_sintax.ipynb
#
# For each input FASTA:
#
#   sample.fasta
#
# MetaDiv_sintax generates, in the same folder:
#
#   sample_sintax.txt
#
# The original FASTA is never modified.
# ============================================================

from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import shlex


# ============================================================
# INTERNAL HELPERS
# ============================================================

FASTA_EXTENSIONS = {
    ".fasta",
    ".fa",
    ".fas",
    ".fna",
    ".ffn",
}


def normalize_mode(mode):
    """
    Normalize the marker name used by MetaDiv Builder.
    """
    value = str(mode).strip().upper()

    aliases = {
        "ITS": "ITS",
        "16S": "16S",
        "COI": "COI",
        "CO1": "COI",
    }

    if value not in aliases:
        raise ValueError(
            "MODE must be one of: ITS, 16S, COI."
        )

    return aliases[value]


def find_project_root():
    """
    Detect the MetaDiv Builder project root automatically.

    The search starts from the current Jupyter working directory
    and moves upward until both of the following are found:

        input/
        databases/taxonomic_reference_db/
    """
    current = Path.cwd().resolve()

    for candidate in [current] + list(current.parents):

        input_dir = candidate / "input"

        reference_dir = (
            candidate
            / "databases"
            / "taxonomic_reference_db"
        )

        if input_dir.is_dir() and reference_dir.is_dir():
            return candidate

    raise FileNotFoundError(
        "\nMetaDiv Builder project root could not be detected.\n\n"
        "The notebook should be located somewhere inside a project "
        "containing:\n\n"
        "  input/\n"
        "  databases/taxonomic_reference_db/\n"
    )


def locate_marker_input(project_root, mode):
    """
    Locate the normal MetaDiv input folder for the selected marker.
    """
    candidates = [
        project_root / "input" / mode,
    ]

    # Backward compatibility with repositories using CO1.
    if mode == "COI":
        candidates.append(
            project_root / "input" / "CO1"
        )

    for directory in candidates:
        if directory.is_dir():
            return directory

    expected = project_root / "input" / mode

    raise FileNotFoundError(
        f"\nMarker input directory not found:\n{expected}\n"
    )


def resolve_input_fastas(input_dir, file_names):
    """
    Resolve and validate the FASTA filenames selected by the user.
    """
    if not isinstance(file_names, (list, tuple)) or len(file_names) == 0:
        raise ValueError(
            "FASTA_FILES must contain at least one filename."
        )

    resolved = []

    for name in file_names:

        fasta = (input_dir / name).resolve()

        try:
            fasta.relative_to(input_dir.resolve())
        except ValueError:
            raise ValueError(
                f"FASTA file must be located inside:\n{input_dir}\n\n"
                f"Invalid path: {name}"
            )

        if not fasta.is_file():
            raise FileNotFoundError(
                f"\nFASTA file not found:\n{fasta}"
            )

        if fasta.suffix.lower() not in FASTA_EXTENSIONS:
            raise ValueError(
                f"\nUnsupported FASTA extension:\n{fasta.name}\n\n"
                f"Accepted extensions: {sorted(FASTA_EXTENSIONS)}"
            )

        resolved.append(fasta)

    return resolved


def output_path_for_fasta(fasta_file):
    """
    Create the companion SINTAX output filename.

    Example:
        ATLASMXB.fasta
        -> ATLASMXB_sintax.txt
    """
    return fasta_file.with_name(
        f"{fasta_file.stem}_sintax.txt"
    )


def pretty_command(command):
    """
    Format a command for the processing report.
    """
    try:
        return shlex.join([str(x) for x in command])
    except Exception:
        return " ".join(str(x) for x in command)


def run_command(
    command,
    check=True,
    capture_output=True,
):
    """
    Execute a system command without using a shell.
    """
    result = subprocess.run(
        [str(x) for x in command],
        check=False,
        text=True,
        capture_output=capture_output,
    )

    if check and result.returncode != 0:

        message = [
            f"Command failed with exit code {result.returncode}.",
            "",
            "Command:",
            pretty_command(command),
        ]

        if result.stdout:
            message.extend(
                [
                    "",
                    "STDOUT:",
                    result.stdout,
                ]
            )

        if result.stderr:
            message.extend(
                [
                    "",
                    "STDERR:",
                    result.stderr,
                ]
            )

        raise RuntimeError(
            "\n".join(message)
        )

    return result


def count_fasta_sequences(fasta_file):
    """
    Count FASTA records using header lines beginning with '>'.
    """
    count = 0

    with fasta_file.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:

        for line in handle:

            if line.startswith(">"):
                count += 1

    return count


def validate_sintax_output(output_file):
    """
    Summarize the raw VSEARCH --tabbedout output.

    Column 1: query identifier
    Column 2: taxonomic prediction with bootstrap support
    Column 3: strand
    Column 4: cutoff-filtered taxonomy when available
    """
    total_records = 0
    taxonomy_predictions = 0
    cutoff_predictions = 0

    with output_file.open(
        "r",
        encoding="utf-8",
        errors="replace",
    ) as handle:

        for line in handle:

            line = line.rstrip("\r\n")

            if not line:
                continue

            fields = line.split("\t")

            total_records += 1

            if (
                len(fields) >= 2
                and fields[1].strip() not in {"", "*"}
            ):
                taxonomy_predictions += 1

            if (
                len(fields) >= 4
                and fields[3].strip() not in {"", "*"}
            ):
                cutoff_predictions += 1

    return {
        "records": total_records,
        "predictions": taxonomy_predictions,
        "cutoff_predictions": cutoff_predictions,
    }


# ============================================================
# VALIDATE USER SETTINGS
# ============================================================

MODE = normalize_mode(MODE)

if not 0.0 <= float(SINTAX_CUTOFF) <= 1.0:
    raise ValueError(
        "SINTAX_CUTOFF must be between 0.0 and 1.0."
    )

if STRAND not in {"plus", "both"}:
    raise ValueError(
        "STRAND must be 'plus' or 'both'."
    )

if int(THREADS) < 1:
    raise ValueError(
        "THREADS must be >= 1."
    )


# ============================================================
# DETECT METADIV PATHS
# ============================================================

PROJECT_ROOT = find_project_root()

INPUT_DIR = locate_marker_input(
    PROJECT_ROOT,
    MODE,
)

REFERENCE_DB_DIR = (
    PROJECT_ROOT
    / "databases"
    / "taxonomic_reference_db"
)

REFERENCE_DB = (
    REFERENCE_DB_DIR
    / REFERENCE_DB_NAME
).resolve()


if not REFERENCE_DB.is_file():
    raise FileNotFoundError(
        "\nReference database not found:\n"
        f"{REFERENCE_DB}\n\n"
        "Check REFERENCE_DB_NAME in the first cell."
    )


INPUT_FASTAS = resolve_input_fastas(
    INPUT_DIR,
    FASTA_FILES,
)


# ============================================================
# CHECK DOCKER DESKTOP
# ============================================================

if shutil.which("docker") is None:
    raise RuntimeError(
        "\nDocker CLI was not detected.\n\n"
        "Install/start Docker Desktop and make sure the Docker "
        "command is available from this Jupyter environment."
    )


docker_info = run_command(
    ["docker", "info"],
    check=False,
)


if docker_info.returncode != 0:
    raise RuntimeError(
        "\nDocker was detected, but Docker Desktop/Engine "
        "does not appear to be running.\n"
    )


# ============================================================
# CHECK / PULL VSEARCH IMAGE
# ============================================================

image_check = run_command(
    [
        "docker",
        "image",
        "inspect",
        DOCKER_IMAGE,
    ],
    check=False,
)


if image_check.returncode != 0:

    if not AUTO_PULL_IMAGE:
        raise RuntimeError(
            f"\nDocker image not found locally:\n{DOCKER_IMAGE}"
        )

    print(
        f"\nDocker image not found locally.\n"
        f"Pulling {DOCKER_IMAGE} ..."
    )

    run_command(
        [
            "docker",
            "pull",
            DOCKER_IMAGE,
        ],
        capture_output=False,
    )


# ============================================================
# VERIFY VSEARCH
# ============================================================

version_result = run_command(
    [
        "docker",
        "run",
        "--rm",
        "--entrypoint",
        "vsearch",
        DOCKER_IMAGE,
        "--version",
    ]
)


VSEARCH_VERSION = (
    version_result.stdout
    or version_result.stderr
    or "unknown"
).strip()


# ============================================================
# RUN SINTAX
# ============================================================

print("=" * 72)
print("MetaDiv_sintax")
print("=" * 72)
print(f"MODE: {MODE}")
print(f"Docker image: {DOCKER_IMAGE}")
print(f"Reference database: {REFERENCE_DB.name}")
print(f"SINTAX cutoff: {SINTAX_CUTOFF}")
print(f"Strand: {STRAND}")
print(f"Threads: {THREADS}")
print(f"Input directory: {INPUT_DIR}")
print(f"FASTA files selected: {len(INPUT_FASTAS)}")
print("=" * 72)


run_started = datetime.now()

RUN_SUMMARY = []


for fasta_file in INPUT_FASTAS:

    output_file = output_path_for_fasta(
        fasta_file
    )

    # --------------------------------------------------------
    # Skip existing outputs unless overwrite is enabled
    # --------------------------------------------------------

    if output_file.exists() and not OVERWRITE_EXISTING:

        print("\n" + "-" * 72)
        print(f"SKIPPED: {fasta_file.name}")
        print(f"Existing output: {output_file.name}")
        print(
            "Set OVERWRITE_EXISTING = True in the first cell "
            "to classify this file again."
        )
        print("-" * 72)

        RUN_SUMMARY.append(
            {
                "input": fasta_file,
                "output": output_file,
                "status": "SKIPPED_EXISTING",
                "sequences": count_fasta_sequences(fasta_file),
                "seconds": 0.0,
                "validation": validate_sintax_output(output_file),
                "command": None,
            }
        )

        continue


    # --------------------------------------------------------
    # Docker mounts
    #
    # /work = folder containing the FASTA and output
    # /db   = taxonomic_reference_db (read only)
    # --------------------------------------------------------

    working_directory = fasta_file.parent

    container_fasta = (
        f"/work/{fasta_file.name}"
    )

    container_output = (
        f"/work/{output_file.name}"
    )

    container_database = (
        f"/db/{REFERENCE_DB.name}"
    )


    command = [
        "docker",
        "run",
        "--rm",

        "--mount",
        (
            "type=bind,"
            f"source={working_directory},"
            "target=/work"
        ),

        "--mount",
        (
            "type=bind,"
            f"source={REFERENCE_DB_DIR},"
            "target=/db,"
            "readonly"
        ),

        "--entrypoint",
        "vsearch",

        DOCKER_IMAGE,

        "--sintax",
        container_fasta,

        "--db",
        container_database,

        "--tabbedout",
        container_output,

        "--sintax_cutoff",
        str(SINTAX_CUTOFF),

        "--strand",
        STRAND,

        "--threads",
        str(THREADS),
    ]


    sequence_count = count_fasta_sequences(
        fasta_file
    )


    print("\n" + "-" * 72)
    print(f"Classifying: {fasta_file.name}")
    print(f"Representative sequences: {sequence_count:,}")
    print(f"Output: {output_file.name}")
    print("-" * 72)


    start_time = datetime.now()

    run_command(
        command,
        check=True,
    )

    end_time = datetime.now()

    elapsed = (
        end_time - start_time
    ).total_seconds()


    if not output_file.is_file():
        raise RuntimeError(
            "\nVSEARCH completed but the expected SINTAX output "
            "was not created:\n"
            f"{output_file}"
        )


    validation = validate_sintax_output(
        output_file
    )


    print(f"SINTAX records: {validation['records']:,}")
    print(
        f"Taxonomy predictions: "
        f"{validation['predictions']:,}"
    )
    print(
        f"Cutoff-filtered predictions: "
        f"{validation['cutoff_predictions']:,}"
    )
    print(f"Execution time: {elapsed:.2f} s")


    RUN_SUMMARY.append(
        {
            "input": fasta_file,
            "output": output_file,
            "status": "CLASSIFIED",
            "sequences": sequence_count,
            "seconds": elapsed,
            "validation": validation,
            "command": command,
        }
    )


run_finished = datetime.now()

TOTAL_SECONDS = (
    run_finished - run_started
).total_seconds()


# ============================================================
# WRITE PROCESSING REPORT
# ============================================================

# The report is written beside the notebook when possible.
REPORT_FILE = (
    Path.cwd()
    / "MetaDiv_sintax_report.txt"
)


report = [
    "=" * 72,
    "MetaDiv_sintax - Processing Report",
    "Author: Bernardo Águila, UNAM",
    f"Date: {run_finished.isoformat(timespec='seconds')}",
    "=" * 72,
    f"MODE: {MODE}",
    f"Docker image: {DOCKER_IMAGE}",
    f"VSEARCH version: {VSEARCH_VERSION}",
    f"Reference database: {REFERENCE_DB}",
    f"SINTAX_CUTOFF: {SINTAX_CUTOFF}",
    f"STRAND: {STRAND}",
    f"THREADS: {THREADS}",
    f"Input directory: {INPUT_DIR}",
    f"FASTA files selected: {len(INPUT_FASTAS)}",
    f"Total execution time (s): {TOTAL_SECONDS:.2f}",
    "=" * 72,
]


for number, item in enumerate(
    RUN_SUMMARY,
    start=1,
):

    report.extend(
        [
            "",
            f"FILE {number}",
            f"Input FASTA: {item['input']}",
            f"Output taxonomy: {item['output']}",
            f"Status: {item['status']}",
            f"Representative sequences: {item['sequences']}",
            (
                "SINTAX records: "
                f"{item['validation']['records']}"
            ),
            (
                "Taxonomy predictions: "
                f"{item['validation']['predictions']}"
            ),
            (
                "Cutoff-filtered predictions: "
                f"{item['validation']['cutoff_predictions']}"
            ),
            f"Execution time (s): {item['seconds']:.2f}",
        ]
    )

    if item["command"] is not None:

        report.extend(
            [
                "Command:",
                pretty_command(
                    item["command"]
                ),
            ]
        )


report.extend(
    [
        "",
        "=" * 72,
        "MetaDiv_sintax completed successfully",
        "=" * 72,
    ]
)


REPORT_FILE.write_text(
    "\n".join(report) + "\n",
    encoding="utf-8",
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 72)
print("MetaDiv_sintax completed successfully")
print("=" * 72)

for item in RUN_SUMMARY:

    print(
        f"{item['input'].name}"
        f"  ->  "
        f"{item['output'].name}"
        f"  [{item['status']}]"
    )

print(
    f"\nTotal execution time: "
    f"{TOTAL_SECONDS:.2f} s"
)

print(
    f"\nProcessing report:\n"
    f"{REPORT_FILE}"
)

print("=" * 72)


MetaDiv_sintax
MODE: ITS
Docker image: pipecraft/vsearch:2.30.4-pc1.2.0
Reference database: SINTAX_EUK_ITS_v2.0.fasta
SINTAX cutoff: 0.8
Strand: both
Threads: 16
Input directory: C:\Users\berna\Desktop\PAPER METADIV\V1_7_13_RUNS\run00 small MAIN\input\ITS
FASTA files selected: 1

------------------------------------------------------------------------
Classifying: ATLASMXBC_red_sequences.fasta
Representative sequences: 82,000
Output: ATLASMXBC_red_sequences_sintax.txt
------------------------------------------------------------------------
